<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/01_Pruning_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# __Pruning 실습__

이번 실습에서는 **Unstructured Pruning**과 **Stuructured Pruning**을 적용하는 방식을 알아봅니다.

# 0. Colab 환경설정
- colab에서 GPU를 사용할 수 있도록 세팅
    - 런타임 > 런타임 유형 변경 > Python 3 와 T4 GPU 선택

## 0-1. Setup
필요한 package를 설치 및 import 합니다.

In [ ]:
! pip install torch_pruning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.9 MB/s eta 0:00:00


In [ ]:
import torch
from torch import nn
import torch.nn.utils.prune as prune
import torch.nn.functional as F
import matplotlib.pyplot as plt
# matplotlib 오류방지
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 1. 가지치기 기법(Pruning) 튜토리얼


이번 튜토리얼에서는, ``torch.nn.utils.prune`` 을 사용하여 여러분이 설계한 딥러닝 모델에 대해 가지치기 기법을 적용해보는 것을 배워보고,
심화적으로 여러분의 맞춤형 가지치기 기법을 구현하는 방법에 대해 배우는 것을 목표로 합니다.



## 1-1. 딥러닝 모델 생성

이번 튜토리얼에서는, 얀 르쿤 교수님의 연구진들이 1998년도에 발표한 [LeNet](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf) 의 모델 구조를 이용합니다.



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # 1개 채널 수의 이미지를 입력값으로 이용하여 6개 채널 수의 출력값을 계산하는 방식
        # Convolution 연산을 진행하는 커널(필터)의 크기는 5x5 을 이용
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  ###### [실습] Convolution 연산 결과 5x5 크기의 16 채널 수의 이미지가 입력됩니다. ######
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, int(x.nelement() / x.shape[0]))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = LeNet().to(device=device)

## 1-2. 모듈 점검

가지치기 기법이 적용되지 않은 LeNet 모델의 ``conv1`` 층을 점검해봅시다.
여기에는 2개의 파라미터값들인 ``가중치`` 값과 ``편향`` 값이 포함될 것이며, 버퍼는 존재하지 않을 것입니다.




In [ ]:
module = model.conv1
print(list(module.named_parameters()))

[('weight', Parameter containing:
tensor([[[[ 0.0019, -0.0069,  0.1850,  0.1719, -0.0779],
          [-0.0010,  0.0070,  0.1628,  0.0038, -0.0452],
          [ 0.1015,  0.1459,  0.0109,  0.0986,  0.1335],
          [-0.0787,  0.1875,  0.0408,  0.1935, -0.0072],
          [ 0.1886,  0.0118,  0.0239,  0.1494, -0.1710]]],


        [[[ 0.0387,  0.0665, -0.0119,  0.0947,  0.0834],
          [-0.1324,  0.0065, -0.0709,  0.1961, -0.0135],
          [-0.0814, -0.1514, -0.1173, -0.1782,  0.1418],
          [-0.0416, -0.0911,  0.0344, -0.1487, -0.0897],
          [-0.1399, -0.1548,  0.0318, -0.1029, -0.1450]]],


        [[[ 0.0750,  0.1542, -0.0388,  0.1586, -0.0012],
          [-0.1904,  0.1919, -0.1468, -0.0790, -0.0489],
          [-0.1150, -0.1694,  0.0271, -0.1740,  0.0792],
          [ 0.0257, -0.1254,  0.0237,  0.0822, -0.1899],
          [ 0.0167,  0.0457, -0.0907,  0.1439,  0.0448]]],


        [[[-0.0387,  0.1004, -0.0298,  0.1819, -0.0036],
          [ 0.1919,  0.0217,  0.0925, -0.1

In [ ]:
print(list(module.named_buffers()))

[]


## 1-3. 모듈 가지치기 기법 적용 예제

- 모듈에 대해 가지치기 기법을 적용하기 위해 (이번 예제에서는, LeNet 모델의 ``conv1`` 층)
첫 번째로는, ``torch.nn.utils.prune`` (또는 ``BasePruningMethod`` 의 서브 클래스로 직접
[구현](torch-nn-utils-prune) )
내 존재하는 가지치기 기법을 선택합니다.
- 그 후, 해당 모듈 내에서 가지치기 기법을 적용하고자 하는 모듈과 파라미터를 지정합니다.
- 마지막으로, 가지치기 기법에 적당한 키워드 인자값을 이용하여 가지치기 매개변수를 지정합니다.


- 이번 예제에서는, ``conv1`` 층의 가중치의 30%값들을 랜덤으로 가지치기 기법을 적용해보겠습니다.
- 모듈은 함수에 대한 첫 번째 인자값으로 전달되며, ``name`` 은 문자열 식별자를 이용하여 해당 모듈 내 매개변수를 구분합니다.
- ``amount`` 는 가지치기 기법을 적용하기 위한 대상 가중치값들의 백분율 (0과 1사이의 실수값),
혹은 가중치값의 연결의 개수 (음수가 아닌 정수) 를 지정합니다.


In [ ]:
###### [실습] prune 패키지의 random_unstructured 함수를 사용하세요. ######
# 다음은 해당 함수의 안내입니다. https://pytorch.org/docs/stable/generated/torch.nn.utils.prune.random_unstructured.html
prune.random_unstructured(module, name="weight", amount=0.3)

Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))

- 가지치기 기법은 가중치값들을 파라미터값들로부터 제거하고 ``weight_orig`` (즉, 초기 가중치 이름에 "_orig"을 붙인) 이라는
새로운 파라미터값으로 대체하는 것으로 실행됩니다.
- ``weight_orig`` : 텐서값에 가지치기 기법이 적용되지 않은 상태를 저장합니다.
- ``bias`` : 가지치기 기법이 적용되지 않았기 때문에 그대로 남아 있습니다.



In [ ]:
print(list(module.named_parameters()))

[('weight_orig', Parameter containing:
tensor([[[[ 0.0019, -0.0069,  0.1850,  0.1719, -0.0779],
          [-0.0010,  0.0070,  0.1628,  0.0038, -0.0452],
          [ 0.1015,  0.1459,  0.0109,  0.0986,  0.1335],
          [-0.0787,  0.1875,  0.0408,  0.1935, -0.0072],
          [ 0.1886,  0.0118,  0.0239,  0.1494, -0.1710]]],


        [[[ 0.0387,  0.0665, -0.0119,  0.0947,  0.0834],
          [-0.1324,  0.0065, -0.0709,  0.1961, -0.0135],
          [-0.0814, -0.1514, -0.1173, -0.1782,  0.1418],
          [-0.0416, -0.0911,  0.0344, -0.1487, -0.0897],
          [-0.1399, -0.1548,  0.0318, -0.1029, -0.1450]]],


        [[[ 0.0750,  0.1542, -0.0388,  0.1586, -0.0012],
          [-0.1904,  0.1919, -0.1468, -0.0790, -0.0489],
          [-0.1150, -0.1694,  0.0271, -0.1740,  0.0792],
          [ 0.0257, -0.1254,  0.0237,  0.0822, -0.1899],
          [ 0.0167,  0.0457, -0.0907,  0.1439,  0.0448]]],


        [[[-0.0387,  0.1004, -0.0298,  0.1819, -0.0036],
          [ 0.1919,  0.0217,  0.0925,

위에서 선택한 가지치기 기법에 의해 생성되는 가지치기 마스크는 초기 파라미터  ``name`` 에 ``weight_mask``
(즉, 초기 가중치 이름에 "_mask"를 붙인) 이름의 모듈 버퍼로 저장됩니다.



In [ ]:
print(list(module.named_buffers()))

[('weight_mask', tensor([[[[0., 0., 0., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 0., 1., 0., 1.],
          [1., 1., 1., 1., 1.],
          [0., 1., 1., 1., 1.]]],


        [[[1., 0., 1., 1., 1.],
          [1., 0., 1., 0., 0.],
          [1., 0., 1., 0., 1.],
          [0., 1., 0., 1., 1.],
          [1., 1., 1., 1., 0.]]],


        [[[0., 1., 1., 1., 0.],
          [1., 1., 0., 1., 1.],
          [1., 1., 0., 1., 1.],
          [0., 1., 1., 1., 1.],
          [1., 1., 0., 0., 0.]]],


        [[[0., 0., 1., 1., 1.],
          [1., 0., 1., 0., 0.],
          [0., 1., 1., 1., 1.],
          [1., 0., 1., 0., 1.],
          [0., 1., 0., 1., 1.]]],


        [[[1., 1., 1., 0., 1.],
          [1., 1., 1., 1., 0.],
          [0., 1., 0., 0., 1.],
          [1., 1., 0., 1., 1.],
          [0., 1., 1., 1., 1.]]],


        [[[1., 1., 0., 1., 1.],
          [1., 1., 1., 0., 1.],
          [0., 1., 0., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 0.]]]], 

- 수정이 되지 않은 상태에서 순전파를 진행하기 위해서는 ``가중치`` 값 속성이 존재해야 합니다.
- ``torch.nn.utils.prune`` 내 구현된 가지치기 기법은 가지치기 기법이 적용된 가중치값들을 이용하여
(기존의 가중치값에 가지치기 기법이 적용된) 순전파를 진행하고, ``weight`` 속성값에 가지치기 기법이 적용된 가중치값들을 저장합니다.
- 이제 가중치값들은 ``module`` 의 매개변수가 아니라 하나의 속성값으로 취급되는 점을 주의하세요.



최종적으로, 가지치기 기법은 파이토치의 ``forward_pre_hooks`` 를 이용하여 각 순전파가 진행되기 전에 가지치기 기법이 적용됩니다.




구체적으로, 지금까지 진행한 것 처럼, 모듈이 가지치기 기법이 적용되었을 때,
가지치기 기법이 적용된 각 파라미터값들이 ``forward_pre_hook`` 를 얻게됩니다.

이러한 경우, ``weight`` 이름인 기존 파라미터값에 대해서만 가지치기 기법을 적용하였기 때문에,
훅은 오직 1개만 존재할 것입니다.

In [ ]:
print(module._forward_pre_hooks)

OrderedDict({0: <torch.nn.utils.prune.RandomUnstructured object at 0x78062d01c8c0>})


- 완결성을 위해, 편향값에 대해서도 가지치기 기법을 적용할 수 있으며,
모듈의 파라미터, 버퍼, 훅, 속성값들이 어떻게 변경되는지 확인할 수 있습니다.
- 또 다른 가지치기 기법을 적용해보기 위해, ``l1_unstructured`` 가지치기 함수에서 구현된 내용과 같이,
L1 Norm 값이 가장 작은 편향값(bias) 3개를 가지치기를 시도해봅시다.



In [ ]:
###### [실습] 가중치에 적용할 때 "weight"를 사용했습니다. 이번에는 어떻게 해야 할까요? ######
prune.l1_unstructured(module, name="bias", amount=3)

Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))

- 이전에서 실습한 내용을 토대로, 명명된 파라미터값들이 ``weight_orig``, ``bias_orig`` 2개를 모두 포함할 것이라 예상할 수 있습니다.
- 버퍼들은 ``weight_mask``, ``bias_mask`` 2개를 포함할 것입니다.
- 가지치기 기법이 적용된 2개의 텐서값들은 모듈의 속성값으로 존재할 것이며, 모듈은 2개의 ``forward_pre_hooks`` 을 갖게 될 것입니다.



In [ ]:
print(list(module.named_parameters()))

[('weight_orig', Parameter containing:
tensor([[[[ 0.0019, -0.0069,  0.1850,  0.1719, -0.0779],
          [-0.0010,  0.0070,  0.1628,  0.0038, -0.0452],
          [ 0.1015,  0.1459,  0.0109,  0.0986,  0.1335],
          [-0.0787,  0.1875,  0.0408,  0.1935, -0.0072],
          [ 0.1886,  0.0118,  0.0239,  0.1494, -0.1710]]],


        [[[ 0.0387,  0.0665, -0.0119,  0.0947,  0.0834],
          [-0.1324,  0.0065, -0.0709,  0.1961, -0.0135],
          [-0.0814, -0.1514, -0.1173, -0.1782,  0.1418],
          [-0.0416, -0.0911,  0.0344, -0.1487, -0.0897],
          [-0.1399, -0.1548,  0.0318, -0.1029, -0.1450]]],


        [[[ 0.0750,  0.1542, -0.0388,  0.1586, -0.0012],
          [-0.1904,  0.1919, -0.1468, -0.0790, -0.0489],
          [-0.1150, -0.1694,  0.0271, -0.1740,  0.0792],
          [ 0.0257, -0.1254,  0.0237,  0.0822, -0.1899],
          [ 0.0167,  0.0457, -0.0907,  0.1439,  0.0448]]],


        [[[-0.0387,  0.1004, -0.0298,  0.1819, -0.0036],
          [ 0.1919,  0.0217,  0.0925,

In [ ]:
print(list(module.named_buffers()))

[('weight_mask', tensor([[[[0., 0., 0., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 0., 1., 0., 1.],
          [1., 1., 1., 1., 1.],
          [0., 1., 1., 1., 1.]]],


        [[[1., 0., 1., 1., 1.],
          [1., 0., 1., 0., 0.],
          [1., 0., 1., 0., 1.],
          [0., 1., 0., 1., 1.],
          [1., 1., 1., 1., 0.]]],


        [[[0., 1., 1., 1., 0.],
          [1., 1., 0., 1., 1.],
          [1., 1., 0., 1., 1.],
          [0., 1., 1., 1., 1.],
          [1., 1., 0., 0., 0.]]],


        [[[0., 0., 1., 1., 1.],
          [1., 0., 1., 0., 0.],
          [0., 1., 1., 1., 1.],
          [1., 0., 1., 0., 1.],
          [0., 1., 0., 1., 1.]]],


        [[[1., 1., 1., 0., 1.],
          [1., 1., 1., 1., 0.],
          [0., 1., 0., 0., 1.],
          [1., 1., 0., 1., 1.],
          [0., 1., 1., 1., 1.]]],


        [[[1., 1., 0., 1., 1.],
          [1., 1., 1., 0., 1.],
          [0., 1., 0., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 0.]]]], 

In [ ]:
print(module.bias)

tensor([ 0.0000,  0.1497, -0.0000, -0.1566, -0.1955, -0.0000], device='cuda:0',
       grad_fn=<MulBackward0>)


In [ ]:
print(module._forward_pre_hooks)

OrderedDict({0: <torch.nn.utils.prune.RandomUnstructured object at 0x78062d01c8c0>, 1: <torch.nn.utils.prune.L1Unstructured object at 0x78062c8b5ac0>})


## 1-4. 가지치기 기법의 재-파라미터화 제거

- 가지치기 기법이 적용된 것을 영구적으로 만들기 위해서, 재-파라미터화 관점의
``weight_orig`` 와 ``weight_mask`` 값을 제거하고, ``forward_pre_hook`` 값을 제거합니다.
- 제거하기 위해 ``torch.nn.utils.prune`` 내 ``remove`` 함수를 이용할 수 있습니다.
- 가지치기 기법이 적용되지 않은 것처럼 실행되는 것이 아닌 점을 주의하세요.
- 이는 단지 가지치기 기법이 적용된 상태에서 가중치 파라미터값을 모델 파라미터값으로 재할당하는 것을 통해 영구적으로 만드는 것일 뿐입니다.



### 1-4-1. 재-파라미터화를 제거하기 전 상태



In [ ]:
print(list(module.named_parameters()))

[('weight_orig', Parameter containing:
tensor([[[[ 0.0019, -0.0069,  0.1850,  0.1719, -0.0779],
          [-0.0010,  0.0070,  0.1628,  0.0038, -0.0452],
          [ 0.1015,  0.1459,  0.0109,  0.0986,  0.1335],
          [-0.0787,  0.1875,  0.0408,  0.1935, -0.0072],
          [ 0.1886,  0.0118,  0.0239,  0.1494, -0.1710]]],


        [[[ 0.0387,  0.0665, -0.0119,  0.0947,  0.0834],
          [-0.1324,  0.0065, -0.0709,  0.1961, -0.0135],
          [-0.0814, -0.1514, -0.1173, -0.1782,  0.1418],
          [-0.0416, -0.0911,  0.0344, -0.1487, -0.0897],
          [-0.1399, -0.1548,  0.0318, -0.1029, -0.1450]]],


        [[[ 0.0750,  0.1542, -0.0388,  0.1586, -0.0012],
          [-0.1904,  0.1919, -0.1468, -0.0790, -0.0489],
          [-0.1150, -0.1694,  0.0271, -0.1740,  0.0792],
          [ 0.0257, -0.1254,  0.0237,  0.0822, -0.1899],
          [ 0.0167,  0.0457, -0.0907,  0.1439,  0.0448]]],


        [[[-0.0387,  0.1004, -0.0298,  0.1819, -0.0036],
          [ 0.1919,  0.0217,  0.0925,

In [ ]:
print(list(module.named_buffers()))

[('weight_mask', tensor([[[[0., 0., 0., 1., 1.],
          [1., 0., 1., 1., 1.],
          [0., 0., 0., 0., 0.],
          [1., 1., 1., 0., 1.],
          [0., 0., 0., 1., 1.]]],


        [[[0., 0., 0., 1., 0.],
          [1., 0., 1., 0., 0.],
          [1., 0., 1., 0., 0.],
          [0., 1., 0., 0., 0.],
          [0., 0., 1., 1., 0.]]],


        [[[0., 1., 1., 1., 0.],
          [0., 0., 0., 1., 0.],
          [1., 1., 0., 1., 1.],
          [0., 1., 1., 0., 1.],
          [1., 1., 0., 0., 0.]]],


        [[[0., 0., 1., 1., 1.],
          [1., 0., 1., 0., 0.],
          [0., 1., 0., 0., 1.],
          [0., 0., 1., 0., 1.],
          [0., 1., 0., 1., 1.]]],


        [[[1., 0., 1., 0., 1.],
          [0., 0., 1., 1., 0.],
          [0., 1., 0., 0., 1.],
          [1., 1., 0., 1., 1.],
          [0., 1., 1., 1., 1.]]],


        [[[1., 1., 0., 1., 0.],
          [1., 0., 1., 0., 1.],
          [0., 1., 0., 1., 0.],
          [1., 0., 0., 0., 1.],
          [1., 0., 1., 1., 0.]]]], 

In [ ]:
print(module.weight)

tensor([[[[ 0.0000, -0.0000,  0.0000,  0.1719, -0.0779],
          [-0.0010,  0.0000,  0.1628,  0.0038, -0.0452],
          [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
          [-0.0787,  0.1875,  0.0408,  0.0000, -0.0072],
          [ 0.0000,  0.0000,  0.0000,  0.1494, -0.1710]]],


        [[[ 0.0000,  0.0000, -0.0000,  0.0947,  0.0000],
          [-0.1324,  0.0000, -0.0709,  0.0000, -0.0000],
          [-0.0814, -0.0000, -0.1173, -0.0000,  0.0000],
          [-0.0000, -0.0911,  0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000,  0.0318, -0.1029, -0.0000]]],


        [[[ 0.0000,  0.1542, -0.0388,  0.1586, -0.0000],
          [-0.0000,  0.0000, -0.0000, -0.0790, -0.0000],
          [-0.1150, -0.1694,  0.0000, -0.1740,  0.0792],
          [ 0.0000, -0.1254,  0.0237,  0.0000, -0.1899],
          [ 0.0167,  0.0457, -0.0000,  0.0000,  0.0000]]],


        [[[-0.0000,  0.0000, -0.0298,  0.1819, -0.0036],
          [ 0.1919,  0.0000,  0.0925, -0.0000, -0.0000],
          [-0.0000,

### 1-4-2. 재-파라미터를 제거한 후 상태



In [ ]:
###### [실습] module의 마스크에 해당하는 weight 값을 삭제(remove)하세요. ######
prune.remove(module, 'weight')
# 해당 결과값을 출력합니다.
print(list(module.named_parameters()))

[('bias_orig', Parameter containing:
tensor([ 0.1356,  0.1497, -0.1332, -0.1566, -0.1955, -0.0886], device='cuda:0',
       requires_grad=True)), ('weight', Parameter containing:
tensor([[[[ 0.0000, -0.0000,  0.0000,  0.1719, -0.0779],
          [-0.0010,  0.0000,  0.1628,  0.0038, -0.0452],
          [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
          [-0.0787,  0.1875,  0.0408,  0.0000, -0.0072],
          [ 0.0000,  0.0000,  0.0000,  0.1494, -0.1710]]],


        [[[ 0.0000,  0.0000, -0.0000,  0.0947,  0.0000],
          [-0.1324,  0.0000, -0.0709,  0.0000, -0.0000],
          [-0.0814, -0.0000, -0.1173, -0.0000,  0.0000],
          [-0.0000, -0.0911,  0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000,  0.0318, -0.1029, -0.0000]]],


        [[[ 0.0000,  0.1542, -0.0388,  0.1586, -0.0000],
          [-0.0000,  0.0000, -0.0000, -0.0790, -0.0000],
          [-0.1150, -0.1694,  0.0000, -0.1740,  0.0792],
          [ 0.0000, -0.1254,  0.0237,  0.0000, -0.1899],
          [ 0.0

In [ ]:
print(list(module.named_buffers()))

[('bias_mask', tensor([0., 1., 0., 1., 1., 0.], device='cuda:0'))]


## 1-5. 모델 내 여러 파라미터값들에 대하여 가지치기 기법 적용

가지치기 기법을 적용하고 싶은 파라미터값들을 지정함으로써, 이번 예제에서 볼 수 있는 것 처럼,
신경망 모델 내 여러 텐서값들에 대해서 쉽게 가지치기 기법을 적용할 수 있습니다.



In [ ]:
new_model = LeNet()
for name, module in new_model.named_modules():
    # 모든 2D-conv 층의 20% 연결에 대해 가지치기 기법을 적용
    if isinstance(module, torch.nn.Conv2d):
        ###### [실습] l1_unstructured 함수로 20%(0.2) 가지치기를 수행하세요. ######
        prune.l1_unstructured(module, name='weight', amount=0.2)
    # 모든 선형 층의 40% 연결에 대해 가지치기 기법을 적용
    elif isinstance(module, torch.nn.Linear):
        ###### [실습] l1_unstructured 함수로 40%(0.4) 가지치기를 수행하세요. ######
        prune.l1_unstructured(module, name='weight', amount=0.4)

print(dict(new_model.named_buffers()).keys())  # 존재하는 모든 마스크들을 확인

dict_keys(['conv1.weight_mask', 'conv2.weight_mask', 'fc1.weight_mask', 'fc2.weight_mask', 'fc3.weight_mask'])


## 1-6. 전역 범위에 대한 가지치기 기법 적용

- 지금까지, "지역 변수" 에 대해서만 가지치기 기법을 적용하는 방법을 살펴보았습니다.
    - 즉, 가중치 규모, 활성화 정도, 경사값 등의 각 항목의 통계량을 바탕으로 모델 내 텐서값 하나씩 가지치기 기법을 적용하는 방식



- 그러나, 범용적이고 아마 더 강력한 방법은 각 층에서 가장 낮은 20%의 연결을 제거하는 것 대신에, 전체 모델에 대해서 가장 낮은 20% 연결을 한번에 제거하는 것입니다.
- 이것은 각 층에 대해서 가지치기 기법을 적용하는 연결의 백분율값을 다르게 만들 가능성이 있습니다.
- ``torch.nn.utils.prune`` 내 ``global_unstructured`` 을 이용하여 어떻게 전역 범위에 대한 가지치기 기법을 적용하는지 살펴봅시다.

In [ ]:
model = LeNet()

parameters_to_prune = (
    (model.conv1, 'weight'),
    (model.conv2, 'weight'),
    (model.fc1, 'weight'),
    (model.fc2, 'weight'),
    (model.fc3, 'weight'),
)

###### [실습] global_unstructured 함수를 사용해, 전체 모델의 20% 연결에 대해 가지치기 기법을 적용합니다. ######
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,
)

이제 각 층에 존재하는 연결들에 가지치기 기법이 적용된 정도가 20%가 아닌 것을 확인할 수 있습니다.
그러나, 전체 가지치기 적용 범위는 약 20%가 될 것입니다.



In [ ]:
print(
    "Sparsity in conv1.weight: {:.2f}%".format(
        100. * float(torch.sum(model.conv1.weight == 0))
        / float(model.conv1.weight.nelement())
    )
)
print(
    "Sparsity in conv2.weight: {:.2f}%".format(
        100. * float(torch.sum(model.conv2.weight == 0))
        / float(model.conv2.weight.nelement())
    )
)
print(
    "Sparsity in fc1.weight: {:.2f}%".format(
        100. * float(torch.sum(model.fc1.weight == 0))
        / float(model.fc1.weight.nelement())
    )
)
print(
    "Sparsity in fc2.weight: {:.2f}%".format(
        100. * float(torch.sum(model.fc2.weight == 0))
        / float(model.fc2.weight.nelement())
    )
)
print(
    "Sparsity in fc3.weight: {:.2f}%".format(
        100. * float(torch.sum(model.fc3.weight == 0))
        / float(model.fc3.weight.nelement())
    )
)
print(
    "Global sparsity: {:.2f}%".format(
        100. * float(
            torch.sum(model.conv1.weight == 0)
            + torch.sum(model.conv2.weight == 0)
            + torch.sum(model.fc1.weight == 0)
            + torch.sum(model.fc2.weight == 0)
            + torch.sum(model.fc3.weight == 0)
        )
        / float(
            model.conv1.weight.nelement()
            + model.conv2.weight.nelement()
            + model.fc1.weight.nelement()
            + model.fc2.weight.nelement()
            + model.fc3.weight.nelement()
        )
    )
)

Sparsity in conv1.weight: 6.00%
Sparsity in conv2.weight: 12.92%
Sparsity in fc1.weight: 22.33%
Sparsity in fc2.weight: 11.68%
Sparsity in fc3.weight: 9.29%
Global sparsity: 20.00%


# 2. ResNet Pruning

- 아래 실습을 통해서 ResNet18 모델에 **Structured Pruning**을 적용합니다.
- 이를 통해 모델의 파라미터 수를 줄이고, 구조적 경량화를 수행하는 과정을 확인할 수 있습니다.

In [ ]:
import torch
from torchvision.models import resnet18
import torch_pruning as tp

## 2-1. 모델 구조 변화 확인

In [ ]:
###### [실습] resnet18 모델을 미리 학습된 가중치로 불러오고, 평가 모드로 설정합니다. ######
model = resnet18(pretrained=True).eval()

# 1. ResNet18의 Dependency Graph를 생성합니다.
###### [실습] example_inputs : 배치 크기 1, 3채널, 224x224 크기의 이미지를 생성합니다. (torch.randn 사용) ######
DG = tp.DependencyGraph().build_dependency(model, example_inputs=torch.randn(1, 3, 224, 224))

# 2. model.conv1 레이어의 특정 채널 인덱스를 이용해 가지치기 그룹을 가져옵니다.
# 이는 특정 레이어를 pruning할 때 연결된 다른 레이어도 함께 수정되도록 하기 위함입니다.
group = DG.get_pruning_group(model.conv1, tp.prune_conv_out_channels, idxs=[2, 6, 9])

# 그룹 세부 정보를 출력합니다.
print(group.details())

# 3. 가지치기를 실행합니다.
if DG.check_pruning_group(group): # 채널 수가 0이 되는 Over-Pruning를 피합니다.
    group.prune()

# 4. 모델을 저장하고 불러옵니다.
model.zero_grad() # 큰 파일 크기를 방지하기 위해 기울기를 초기화합니다.
torch.save(model, '/content/model.pth')
# 5. 가지치기 된 모델을 torch.load를 사용해 불러옵니다.
model = torch.load('/content/model.pth', weights_only=False)

# 그룹 세부 정보를 출력합니다.
print(group.details()) # 또는 print(group)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]



--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on conv1 (Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)) => prune_out_channels on conv1 (Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)), idxs (3) =[2, 6, 9]  (Pruning Root)
[1] prune_out_channels on conv1 (Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)) => prune_out_channels on bn1 (BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)), idxs (3) =[2, 6, 9] 
[2] prune_out_channels on bn1 (BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)) => prune_out_channels on _ElementWiseOp_20(ReluBackward0), idxs (3) =[2, 6, 9] 
[3] prune_out_channels on _ElementWiseOp_20(ReluBackward0) => prune_out_channels on _ElementWiseOp_19(MaxPool2DWithIndicesBackward0), idxs (3) =[2, 6, 9] 
[4] prune_out_channels on _ElementWiseOp_19(MaxPool2DWithInd

## 2-2. 파라미터 수 변화 확인

In [ ]:
# ResNet18 모델을 미리 학습된 가중치로 불러오고, 평가 모드로 설정합니다.
model = resnet18(pretrained=True).eval()


###### [실습] 가지치기 전의 전체 파라미터 수(numel)를 계산합니다. ######
total_params_before = sum(p.numel() for p in model.parameters())

# 1. ResNet18의 Dependency Graph를 생성합니다.
DG = tp.DependencyGraph().build_dependency(model, example_inputs=torch.randn(1, 3, 224, 224))

###### [실습] 2. model.conv1 레이어의 특정 채널 인덱스를 이용해 가지치기 그룹을 가져오세요. ######
group = DG.get_pruning_group(model.conv1, tp.prune_conv_out_channels, idxs=[0,1,2,3,4,5,6,7,8])

# 3. 가지치기를 실행합니다.
if DG.check_pruning_group(group): # 채널 수가 0이 되는 Over-Pruning를 피합니다.
    group.prune()

# 가지치기 후의 전체 파라미터 수를 계산합니다.
total_params_after = sum(p.numel() for p in model.parameters())

# 결과 출력
print(f"Total parameters before pruning: {total_params_before}")
print(f"Total parameters after pruning: {total_params_after}")

# 그룹 세부 정보를 출력합니다.
print(group.details())

Total parameters before pruning: 11689512
Total parameters after pruning: 11655879

--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on conv1 (Conv2d(3, 55, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)) => prune_out_channels on conv1 (Conv2d(3, 55, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)), idxs (9) =[0, 1, 2, 3, 4, 5, 6, 7, 8]  (Pruning Root)
[1] prune_out_channels on conv1 (Conv2d(3, 55, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)) => prune_out_channels on bn1 (BatchNorm2d(55, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)), idxs (9) =[0, 1, 2, 3, 4, 5, 6, 7, 8] 
[2] prune_out_channels on bn1 (BatchNorm2d(55, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)) => prune_out_channels on _ElementWiseOp_20(ReluBackward0), idxs (9) =[0, 1, 2, 3, 4, 5, 6, 7, 8] 
[3] prune_out_channels on _ElementWiseOp_20(ReluBackward0) => prune_out_channe

**결과 분석**
- 채널 pruning을 통해 모델의 파라미터 수가 감소했음을 확인할 수 있습니다.
- Structured Pruning이기 때문에 실제 연산량 감소 및 추론 속도 개선에도 영향을 줄 수 있습니다.

## 2-3. 실행시간 비교

#### **ResNet18 Pruning 전후 Inference 성능 비교**

- ImageNet으로 사전학습된 ResNet18 모델을 사용하여 pruning 전후의 inference 속도를 비교합니다.

1. 원본 ResNet18 모델을 GPU에서 실행하여 기준 성능을 측정합니다.
2. 이후 ResNet18의 여러 `conv1` 계열 layer에서 output channel의 `pruning ratio`만큼 제거하여 pruned model을 생성합니다.
3. 동일한 입력에 대해 pruning 이후의 inference 성능을 다시 측정합니다.



##### **Benchmark 측정 방식**

- 모델은 총 200번 inference를 수행합니다.
- 이 중 처음 50번은 warmup 단계로 제외합니다.
    - GPU는 초기 실행 시 커널 로딩, cuDNN 알고리즘 선택, 캐시 준비 등의 영향으로 시간이 불안정할 수 있기 때문입니다.
- 따라서 실제 평균 inference time은 나머지 150번의 결과만 사용하여 계산합니다.



##### **CPU와 GPU를 나누어 사용하는 이유**

- pruning 과정은 모델 구조를 수정하는 작업에 가깝기 때문에 CPU에서 수행합니다.
- 반면 benchmark는 실제 inference 속도를 측정하는 과정이므로 GPU에서 수행합니다.

만약 pruning까지 GPU에서 수행하면, pruning 과정에서 생성되는 dependency graph나 임시 tensor들이 benchmark 결과에 영향을 줄 수 있습니다.  
따라서 더 안정적인 benchmark를 위해:

- pruning은 CPU에서 수행하고,
- pruning이 완료된 모델만 GPU로 옮겨 inference benchmark를 수행합니다.



##### **Pruning (CPU에서 수행)**

- pruning은 Conv layer의 일부 output channel을 제거하여 모델 구조를 단순화하는 과정입니다.
- 이 과정에서는 `torch_pruning.DependencyGraph`를 사용하여 layer 간 연결 관계를 분석합니다.
- 특정 channel을 제거하면 연결된 BatchNorm이나 다음 layer의 입력 channel도 함께 수정되어야 합니다.



##### **Benchmark (GPU에서 수행)**

- benchmark 단계에서는 실제 inference 시간을 측정합니다.
- 따라서 모델과 입력 데이터를 GPU로 이동한 뒤 inference를 수행합니다.



##### **비교하는 항목**

- 원본 ResNet18의 평균 inference time
- pruning된 ResNet18의 평균 inference time

In [ ]:
import torch
import torch_pruning as tp
# torchvision에서 제공하는 ResNet18 모델과 pretrained weight를 가져옵니다.
from torchvision.models import resnet18, ResNet18_Weights

# =========================
# 기본 설정
# =========================

# 모델 inference를 총 몇 번 반복할지 정합니다.
num_runs = 200

# 처음 몇 번의 실행 결과를 평균 계산에서 제외할지 정합니다.
# GPU는 초반 실행 때 속도가 불안정할 수 있으므로 warmup을 둡니다.
warmup = 50
device = "cuda"
batch_size = 32

# 입력 크기가 고정되어 있을 때 cuDNN이 빠른 연산 방식을 찾도록 합니다.
torch.backends.cudnn.benchmark = True

# =========================
# 입력 데이터 생성
# =========================

# pruning할 때 사용할 입력입니다.
###### [실습] 더미 입력 데이터 생성: batch_size, 3채널, 224x224 크기의 이미지를 생성합니다. ######
dummy_input_cpu = torch.randn(batch_size, 3, 224, 224)

# benchmark할 때 사용할 입력입니다.
# benchmark는 GPU에서 수행할 것이므로 CPU tensor를 GPU로 옮깁니다.
dummy_input = dummy_input_cpu.to(device)

# =========================
# Pure Model Benchmark
# =========================

# pretrained ResNet18 모델을 불러옵니다.
# 아직 pruning하지 않은 원본 모델입니다.
pure_model = resnet18(
    weights=ResNet18_Weights.IMAGENET1K_V1
).to(device).eval()

# inference 시간을 저장할 리스트입니다.
times = []

# inference에서는 gradient 계산이 필요 없으므로 비활성화합니다.
with torch.no_grad():

    # 총 num_runs번 inference를 반복합니다.
    for i in range(num_runs):

        # GPU 연산 시간을 측정하기 위한 CUDA Event를 만듭니다.
        starter = torch.cuda.Event(enable_timing=True)
        ender = torch.cuda.Event(enable_timing=True)

        # 이전 GPU 작업이 모두 끝날 때까지 기다립니다.
        torch.cuda.synchronize()

        # inference 시작 시점을 기록합니다.
        starter.record()

        # 원본 모델에 입력을 넣고 inference를 수행합니다.
        _ = pure_model(dummy_input)

        # inference 종료 시점을 기록합니다.
        ender.record()

        # GPU 작업이 완전히 끝날 때까지 기다립니다.
        torch.cuda.synchronize()

        # 시작 시점과 종료 시점 사이의 시간을 ms 단위로 계산합니다.
        inference_time = starter.elapsed_time(ender)

        # warmup 이후 결과만 저장합니다.
        if i >= warmup:
            times.append(inference_time)

# warmup을 제외한 inference time 평균을 계산합니다.
avg_time = sum(times) / len(times)

# 원본 모델 benchmark 결과를 출력합니다.
print("===== Pure Model =====")
print(f"Average inference time ({num_runs - warmup} runs): {avg_time:.3f} ms")

# 원본 모델을 GPU 메모리에서 제거합니다.
del pure_model

# PyTorch가 잡고 있던 GPU cache를 비웁니다.
torch.cuda.empty_cache()


# =========================
# Pruned Model
# =========================

# pruning할 ResNet18 모델을 새로 불러옵니다.
# pruning은 CPU에서 수행할 것이므로 GPU로 옮기지 않습니다.
model = resnet18(
    weights=ResNet18_Weights.IMAGENET1K_V1
).eval()

# pruning할 Conv layer 이름 목록입니다.
# ResNet18의 conv1 계열 layer들을 pruning합니다.
target_layer_names = [
    "conv1",
    "layer1.0.conv1",
    "layer1.1.conv1",
    "layer2.0.conv1",
    "layer2.1.conv1",
    "layer3.0.conv1",
    "layer3.1.conv1",
    "layer4.0.conv1",
    "layer4.1.conv1",
]

##### [실습] 각 layer의 output channel 중 n%를 제거합니다.
# pruning_ratio에 따른 inference time 변화를 비교해보세요.
# 추천 시도값: 0.2, 0.5(기본), 0.7
pruning_ratio = 0.5

# target_layer_names에 있는 layer를 하나씩 pruning합니다.
for layer_name in target_layer_names:

    # 문자열 이름으로 실제 layer를 가져옵니다.
    target_layer = model.get_submodule(layer_name)

    # 현재 Conv layer의 output channel 개수를 확인합니다.
    current_out_channels = target_layer.out_channels

    # 제거할 channel 개수를 계산합니다.
    num_prune = int(current_out_channels * pruning_ratio)

    # 제거할 channel index를 정합니다.
    # 여기서는 단순히 앞쪽 channel부터 제거합니다.
    pruning_idxs = list(range(num_prune))

    # DependencyGraph를 생성합니다.
    DG = tp.DependencyGraph().build_dependency(
        model,
        example_inputs=dummy_input_cpu
    )

    # target_layer의 output channel을 제거하기 위한 pruning group을 만듭니다.
    group = DG.get_pruning_group(
        target_layer,
        tp.prune_conv_out_channels,
        idxs=pruning_idxs
    )

    # 이 pruning 작업이 모델 구조상 가능한지 확인합니다.
    if DG.check_pruning_group(group):

        ###### [실습] 그룹에 해당하는 채널들을 가지치기(prune) 합니다. ######
        group.prune()

# pruning 과정에서 사용한 객체들을 삭제합니다.
del DG
del group
del target_layer


# =========================
# Pruned Model Benchmark
# =========================

# pruning이 끝난 모델을 GPU로 옮깁니다.
model = model.to(device).eval()

# pruned model의 inference 시간을 저장할 리스트입니다.
times = []

# inference에서는 gradient 계산이 필요 없으므로 비활성화합니다.
with torch.no_grad():

    # 총 num_runs번 inference를 반복합니다.
    for i in range(num_runs):

        # GPU 연산 시간을 측정하기 위한 CUDA Event를 만듭니다.
        starter = torch.cuda.Event(enable_timing=True)
        ender = torch.cuda.Event(enable_timing=True)

        # 이전 GPU 작업이 모두 끝날 때까지 기다립니다.
        torch.cuda.synchronize()

        # inference 시작 시점을 기록합니다.
        starter.record()

        # pruning된 모델에 입력을 넣고 inference를 수행합니다.
        _ = model(dummy_input)

        # inference 종료 시점을 기록합니다.
        ender.record()

        # GPU 작업이 완전히 끝날 때까지 기다립니다.
        torch.cuda.synchronize()

        # 시작 시점과 종료 시점 사이의 시간을 ms 단위로 계산합니다.
        inference_time = starter.elapsed_time(ender)

        # warmup 이후 결과만 저장합니다.
        if i >= warmup:
            times.append(inference_time)

# warmup을 제외한 inference time 평균을 계산합니다.
avg_time = sum(times) / len(times)

# pruning된 모델 benchmark 결과를 출력합니다.
print(f"\n===== Pruned Model - All conv1 {int(pruning_ratio*100)}% =====")
print(f"Average inference time ({num_runs - warmup} runs): {avg_time:.3f} ms")

# GPU cache를 비웁니다.
torch.cuda.empty_cache()

===== Pure Model =====
Average inference time (150 runs): 30.384 ms

===== Pruned Model - All conv1 50% =====
Average inference time (150 runs): 15.658 ms
